# LIANA analysis for HCA Working Group project
### Author: Marton Olbei, Imperial College London
The motivation for this working-group project is to generate cell-cell communication (CCC) data for the major tissue compartments from the combined object, and visualise how the relative importance of celltypes and ligand-receptor events shift as we move anatomically along the healthy human gut.

In [1]:
import scanpy as sc
import liana as li

In [ ]:
# this function is not necessary, it's an analog of glimpse from dplyr in R

def glimpse(df):
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    for col in df.columns:
        print(f"$ {col} <{df[col].dtype}> {df[col].head().values}")

In [ ]:
# read combined object - make sure to change path

adata = sc.read_h5ad('/Users/molbei/Downloads/20250606_CxG_upload_checkpoint.h5ad')

In [ ]:
glimpse(adata.obs)

Rows: 1497230
Columns: 82
$ sample_id <category> ['Pan_T7935490', 'Pan_T7935490', 'Pan_T7935490', 'Pan_T7935490', 'Pan_T7935490']
Categories (545, object): ['Pan_T7917821', 'Pan_T7917827', 'Pan_T7935487', 'Pan_T7935488', ..., 'NC6_CD3', 'NC6_CD45', 'P1-CO-LP-HLADR+', 'P1-SI-LP-HLADR+']
$ author_cell_type <category> ['unknown', 'unknown', 'unknown', 'unknown', 'unknown']
Categories (386, object): ['A', 'AC prog.', 'ADAMDEC1+ stromal', 'AQP8+ AC', ..., 'tuft cell of colon', 'undiff', 'undiff/early enterocyte', 'unknown']
$ closest_GCA_celltype_x <category> [NaN, NaN, NaN, NaN, NaN]
Categories (92, object): ['B Cell Lineage', 'CD4 T', 'CD8 T', 'Cycling Epithelial Cells', ..., 'Enterocytes Late', 'Secretory Goblet cells Mature ', 'EEC Progenitor', 'Mesothelial']
$ donor_id <category> ['A29_390C', 'A29_390C', 'A29_390C', 'A29_390C', 'A29_390C']
Categories (288, object): ['A29_390C', 'A31_403C', 'A35_423C', 'A37_428C', ..., 'NC4', 'NC5', 'NC6', 'P1_HLADR+']
$ dataset_id <category> ['Domingue

In [ ]:
# pulling out normal tissue only
adata_normal=adata[(adata.obs['disease_ontology_term']=='normal')]

In [ ]:
# deleting original object to save on memory
del adata

In [ ]:
# lsit of tissues, although this will change with updates to the object

list_of_tissues = ['ileal mucosa', 'colonic mucosa', 'ileum',
                  'jejunal epithelium', 'jejunal mucosa', 'colon',
                  'colonic epithelium', 'gastrointestinal system mesentery',
                  'transverse colon', 'sigmoid colon', 'ileal epithelium',
                  'caecum', 'cecum mucosa', 'duodenal mucosa', 'rectum',
                  'ascending colon', 'caecum epithelium', 'duodenum',
                  'mucosa of renal medulla collecting duct', 'small intestine',
                  'descending colon', 'jejunum', 'duodenal epithelium',
                  'vermiform appendix']

In [ ]:
# main function that runs LIANA with (almost) default settings
# LIANA results are saved to .csv to be processed further in R
# trycatch structure is there to catch the tissues with no communication
# verbose flag can be turned off to reduce all the messages but it's useful for debugging
def liana_workflow(subset):
    try:
        adata_small = adata_normal[adata_normal.obs['tissue_ontology_term']==f'{subset}']
        adata_small.var_names = adata_small.var['gene_symbol'].astype(str)
        adata_small.var_names_make_unique()
        print(f'{subset}')
        df = li.mt.rank_aggregate(adata_small,
                         groupby='cell_type_atlas_mapped',
                         resource_name='consensus', 
                         use_raw = False,
                         expr_prop=0.1,
                         verbose=True,
                         inplace=False)
        df.to_csv(f'HCA_{subset}.csv', index=False)
    except:
        print(f'{subset} issue')

In [ ]:
# run function on the tissues
for i in list_of_tissues:
    liana_workflow(i)

ileal mucosa
Using `.X`!


/Users/molbei/Library/Python/3.9/lib/python/site-packages/anndata/_core/anndata.py:402: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


4123 features of mat are empty, they will be removed.
Make sure that normalized counts are passed!


/Users/molbei/Library/Python/3.9/lib/python/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_pipe_utils/_pre.py:160: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


The following cell identities were excluded: Fibroblastic reticular cells, Fibroblasts Submucosal (aka Crypt Support/Trophocytes/S3)
0.03 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 252971 samples and 32230 features


...

/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_liana_pipe.py:294: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_liana_pipe.py:422: RuntimeWarning: overflow encountered in power
/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_liana_pipe.py:415: RuntimeWarning: invalid value encountered in subtract
/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_liana_pipe.py:415: RuntimeWarning: invalid value encountered in subtract
/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_liana_pipe.py:415: RuntimeWarning: invalid value encountered in subtract
/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_liana_pipe.py:415: RuntimeWarning: invalid value encountered in subtract
/Users/molbei/Library/Python/3.9/lib/python/site-packages/liana/method/_liana_pipe.py:415

Running CellPhoneDB


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:02<00:00, 334.80it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR
Running CellChat


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:38<00:00, 25.67it/s]
